In [0]:
-- setup temp table to stage the data before transformation
create or replace temp view raw_unstructure_doc as 
select
  path,
  content
from
  read_files('/Volumes/idp/default/youtube_lesson')

In [0]:
select 
  path, length(content)
from
  raw_unstructure_doc

In [0]:
create or replace temp view parsed_structured_docs as

select
  path,
  ai_parse_document(content) as parsed_content
from
  raw_unstructure_doc

In [0]:
select path, parsed_content from parsed_structured_docs

In [0]:
create or replace temp view structured_tables as 
select 
  path,
  try_cast(e:content as string) as table_html
from
  parsed_structured_docs
lateral view 
  explode(try_cast(parsed_content:document:elements as array<variant>)) t as e
where
  try_cast(e:type as string) = 'table'

In [0]:
select
  *
from
  structured_tables